<div style="text-align:center; font-size:50px; font-weight:700; margin-top:25px;">
Python pour Data Science - Projet - Groupe 4A
</div>

<div style="text-align:center; font-size:40px; font-weight:700; margin-top:25px;">
Prédiction des prix immobiliers
</div>

<br>

<div style="display:flex; justify-content:space-between; margin-top:35px; font-size:20px;">

  <div style="width:45%;">
    <div style="font-size:22px; font-weight:700; margin-bottom:10px;">Étudiants :</div>
    <div>• Yassine MELLOUL</div>
    <div>• Amira BARHOUMI</div>
    <div>• Antoine FOUCART</div>
  </div>

  <div style="width:45%; text-align:right;">
    <div style="font-size:22px; font-weight:700; margin-bottom:10px;">Chargé de TD :</div>
    <div>Julien PRAMIL</div>
  </div>

</div>

<br>
<hr>

# **1 - DESCRIPTION ET PROBLEMATIQUE**

Dans un contexte où les prix immobiliers sont fortement hétérogènes selon les territoires, le marché du logement en France présente de fortes disparités liées à la localisation, à 

l’attractivité des zones et aux dynamiques socio-économiques locales. Cette variabilité rend complexe la compréhension des mécanismes de formation des prix. Ainsi, la problématique 

centrale de ce projet est d’identifier et quantifier les facteurs influençant le prix de l’immobilier à l’échelle des communes. Nous nous concentrons en particulier sur des déterminants 

socio-économiques tels que la densité de population, le taux de chômage, le revenu moyen et le taux de pauvreté, afin d’évaluer leur impact sur les niveaux de prix. Pour cela, nous 

mobilisons deux sources de données principales : la base DVF (Demandes de Valeurs Foncières), qui recense les transactions immobilières en France , et les données socio-économiques de 

l’INSEE . Les jeux de données utilisés sont accessibles respectivement sur data.gouv.fr et insee.fr, via les liens suivants : 
https://www.data.gouv.fr/datasets/demandes-de-valeurs-foncieres et https://www.insee.fr/fr/statistiques/5359146 .

In [ ]:
# IMPORTATION
# modules
import pandas as pd
import numpy as np

# fonctions
from src.data.collect import load__data_url_zip_txt, load_insee_dossier_complet,load_logements_sociaux
from src.data.clean import filter_dvf_columns, compute_prix_m2, preprocess_insee, merge_all,handle_missing_values, remove_outliers
from src.data.stats_desc import univariate_numeric_analysis, carte_france_departements, analyse_bivariée_quant_quant, carte_dep_communes_grid


#pour la modélisation
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
 
from src.model.train import prepare_features, split_data, train_linear_regression, train_random_forest, train_gradient_boosting
from src.model.evaluate import evaluate_model, plot_feature_importance, plot_predictions, plot_residuals
#URL
URL_DVF = "https://static.data.gouv.fr/resources/demandes-de-valeurs-foncieres/20260405-002321/valeursfoncieres-2025.txt.zip"
URL_INSEE = "https://www.insee.fr/fr/statistiques/fichier/5359146/dossier_complet.zip"
URL_LOGEMENTS_SOCIAUX = "https://www.data.gouv.fr/api/1/datasets/r/b0d30277-3a14-4673-a988-2fa6c11e030c"





# **2 - COLLECTE DE DONNEES ET NETTOYAGE**

## **2-1 COLLECTE**

In [ ]:
# collecte de la base dvf
dvf = load__data_url_zip_txt(URL_DVF)

# aperçu des données brutes
summary_dvf = dvf.dtypes.to_frame(name="type")
summary_dvf["nb_valeurs_manquantes"] = dvf.isna().sum()

summary_dvf = summary_dvf.reset_index().rename(columns={"index": "variable"})

n_rows, n_cols = dvf.shape

print(f"Nombre de lignes : {n_rows}")
print(f"Nombre de colonnes : {n_cols}")

print(summary_dvf)
print(dvf.head(2))

La base contient 3,7 millions de lignes et 43 colonnes. On observe que de nombreuses colonnes sont quasi entièrement vides (identifiants, lots, articles CGI). Les colonnes pertinentes pour notre étude sont le prix, la surface, le type de bien et la localisation. On conserve uniquement les ventes de maisons et d'appartements avec un prix et une surface strictement positifs, ce qui réduit la base à environ 1,1 million de transactions.

In [ ]:
# collecte de la base INSEE
insee = load_insee_dossier_complet(URL_INSEE)

# aperçu des données brutes
summary_insee = insee.dtypes.to_frame(name="type")
summary_insee["nb_valeurs_manquantes"] = insee.isna().sum()

summary_insee = summary_insee.reset_index().rename(columns={"index": "variable"})

n_rows, n_cols = insee.shape

print(f"Nombre de lignes : {n_rows}")
print(f"Nombre de colonnes : {n_cols}")

print(summary_insee)
print(insee.head(5))

La base contient 34 988 communes et 7 variables. Le taux de pauvreté (TP6021) présente 30 591 valeurs manquantes sur 34 988 (87%), en raison du secret statistique appliqué aux communes de moins de 1 000 ménages. Cette variable est donc exclue de la modélisation. Le revenu médian (MED21) est au format texte avec des virgules comme séparateur décimal, il nécessite une conversion.

In [ ]:
#chargement de la base des logements sociaux
log_soc=load_logements_sociaux(URL_LOGEMENTS_SOCIAUX)

# aperçu des données brutes

summary_log = log_soc.dtypes.to_frame(name="type")
summary_log["nb_valeurs_manquantes"] = log_soc.isna().sum()

summary_log = summary_log.reset_index().rename(columns={"index": "variable"})

n_rows, n_cols = log_soc.shape

print(f"Nombre de lignes : {n_rows}")
print(f"Nombre de colonnes : {n_cols}")

print(summary_log)
print(log_soc.head(5))

## **2-1 NETTOYAGE**

### **Base DVF**
Le nettoyage des données DVF vise à construire un jeu de données cohérent et exploitable pour la modélisation du prix immobilier.

Dans un premier temps, seules les variables pertinentes sont conservées : prix de transaction, localisation (code postal, commune, département, code commune), caractéristiques du bien (type de local, surface bâtie) et type de mutation. Les variables inutiles ou trop détaillées sont supprimées.

Ensuite, plusieurs transformations sont appliquées :
- conversion du prix en format numérique (suppression des espaces, gestion des virgules),
- harmonisation des types (variables qualitatives en `string`),
- standardisation des codes géographiques (zfill pour obtenir des codes à 2 et 5 chiffres).

Un filtrage est réalisé pour ne conserver que les observations pertinentes :
- uniquement les ventes,
- prix et surface strictement positifs,
- biens de type *Maison* ou *Appartement*.

Les valeurs manquantes sont supprimées afin d’assurer la qualité des données.

Enfin, une variable dérivée **prix au m²** est calculée comme le ratio entre la valeur foncière et la surface bâtie.

Ce processus permet d’obtenir un jeu de données propre, homogène et directement utilisable pour l’analyse et la modélisation.



In [ ]:
# Nettoyage
dvf = filter_dvf_columns(dvf)
# Ajout de la varible prix du metre carré
dvf = compute_prix_m2(dvf)
# aperçu des données tratées
summary_dvf_f = dvf.dtypes.to_frame(name="type")
summary_dvf_f["nb_valeurs_manquantes"] = dvf.isna().sum()

summary_dvf_f = summary_dvf_f.reset_index().rename(columns={"index": "variable"})

n_rows, n_cols = dvf.shape

print(f"Nombre de lignes : {n_rows}")
print(f"Nombre de colonnes : {n_cols}")

print(summary_dvf_f)
print(dvf.head(5))

### **Base INSEE**
La fonction `preprocess_insee` a pour objectif de transformer un DataFrame INSEE brut en un jeu de données propre, standardisé et exploitable pour l’analyse statistique ou la modélisation.

Dans un premier temps, une copie du DataFrame est créée afin d’éviter toute modification directe des données d’origine. Cette étape garantit une approche non destructive du traitement.

Ensuite, les colonnes sont renommées pour améliorer leur lisibilité et leur cohérence. Les noms techniques issus de l’INSEE (comme `CODGEO`, `MED21` ou `TP6021`) sont remplacés par des appellations explicites telles que `code_commune`, `mediane_niveau_vie` ou `taux_pauvrete`. Cette standardisation facilite l’écriture du code et la compréhension des variables.

La fonction procède ensuite à la conversion des types de données. Le code commune et la médiane du niveau de vie sont convertis en type `string`, car ils correspondent à des identifiants ou des valeurs textuelles.

À partir des variables existantes, de nouveaux indicateurs sont créés. Le taux de chômage est calculé en rapportant le nombre de chômeurs de 15 à 64 ans à la population active, ce qui fournit une mesure simple de la pression du chômage. La densité de population est également construite en divisant la population par la superficie ( en $km^2$), permettant d’évaluer le degré de concentration des habitants sur le territoire.

In [ ]:
# Nettoyage
insee = preprocess_insee(insee)

# aperçu des données traitées
summary_insee_f = insee.dtypes.to_frame(name="type")
summary_insee_f["nb_valeurs_manquantes"] = insee.isna().sum()

summary_insee_f = summary_insee_f.reset_index().rename(columns={"index": "variable"})

n_rows, n_cols = insee.shape

print(f"Nombre de lignes : {n_rows}")
print(f"Nombre de colonnes : {n_cols}")

print(summary_insee_f)
print(insee.head(5))

### **Fusion des jeux de données selon code commune**

In [ ]:
# merge

data_final = merge_all(dvf, insee,log_soc)
# aperçu
print(data_final.head(6))
print(data_final.shape)

Les variables retenues pour l’analyse combinent des informations immobilières, géographiques et socio-économiques afin de mieux expliquer les dynamiques du prix au mètre carré. La variable cible est la *valeur_foncière* (en euros), complétée par le *prix_m2* (en euros/m²), qui constitue un indicateur normalisé du marché immobilier. Les caractéristiques descriptives du bien incluent le *type_local* (catégorie qualitative : maison ou appartement), la *surface_reelle_bati* (en m²) et le *nombre_de_pieces_principales* (en unités).

Des variables de localisation et d’identification sont également intégrées, telles que le *code_postal*, le *code_commune* et le *code_departement* (codes administratifs INSEE), ainsi que la *commune* (nom de la ville, variable qualitative). Enfin, plusieurs variables socio-économiques enrichissent l’analyse territoriale : la *population* (nombre d’habitants), la *densite* (habitants/km²), le *taux_chomage* (en proportion ou pourcentage), le *taux_logements_sociaux* (part des logements sociaux), la *mediane_niveau_vie* (en euros annuels) et les indicateurs du marché du travail tels que le *nombre de personnes actives âgées de 15 à 64 ans* et le *nombre de chômeurs 15–64 ans* (effectifs). La variable *superficie* (en km²) caractérise quant à elle l’échelle géographique des communes étudiées.

L’ensemble de ces variables permet de relier les prix immobiliers à la fois aux caractéristiques physiques des biens et aux contextes socio-économiques locaux.

# **3 - ANALYSES DESCRIPTIVES**

## **3-1 - Univariées**

### **Valeur foncière**

In [ ]:
stats = univariate_numeric_analysis(data_final,"valeur_fonciere" )

**Valeurs aberrantes!!!**

La variable *valeur_foncière* présente une distribution fortement asymétrique avec une dispersion extrêmement élevée, comme l’illustre l’écart important entre la moyenne (1 033 616.90) et la médiane (202 900), ainsi qu’un écart-type très élevé (9 937 383.38). Cette différence indique la présence de valeurs extrêmes influentes qui tirent fortement la moyenne vers le haut. On observe également des valeurs aberrantes très élevées (jusqu’à 695 000 000) et très faibles (0.15), qui ne sont pas représentatives des transactions immobilières usuelles et risquent de biaiser les analyses statistiques ainsi que les modèles prédictifs.

Afin de limiter l’impact de ces valeurs extrêmes et de mieux représenter la structure centrale des données, une méthode de filtrage par quantiles a été appliquée. Les observations situées en dehors de l’intervalle [5 %, 95 %] de la distribution ont été supprimées, soit les valeurs inférieures au 5e percentile (q05 = 50 914.50) et supérieures au 95e percentile (q95 = 1 150 000.00). Ce traitement permet de réduire la sensibilité aux outliers tout en conservant la majorité de l’information utile. Il améliore ainsi la robustesse des analyses univariées et la stabilité des modèles statistiques en se concentrant sur le comportement typique du marché immobilier.

In [ ]:
q05 = data_final["valeur_fonciere"].quantile(0.05)
q95 = data_final["valeur_fonciere"].quantile(0.95)
data_final = data_final[data_final["valeur_fonciere"] > q05]
data_final = data_final[data_final["valeur_fonciere"] < q95]

stats1q = univariate_numeric_analysis(data_final,"valeur_fonciere")

### **prix du mètre carré**

In [ ]:
stats2 = univariate_numeric_analysis(data_final,"prix_m2" )

Malgré une première étape de suppression des valeurs extrêmes, la variable *prix_m2* continue de présenter des valeurs aberrantes. En effet, la distribution reste fortement dispersée avec une moyenne (3 824.93) nettement supérieure à la médiane (2 729.17) et un écart-type élevé (5 711.40), ce qui indique une asymétrie persistante et la présence de valeurs extrêmes résiduelles. On observe notamment un maximum encore très élevé (1 129 344.50), largement incohérent avec les ordres de grandeur usuels du marché immobilier, même dans des zones très tendues.

Pour limiter davantage l’influence de ces observations atypiques, une seconde phase de filtrage plus stricte a été appliquée à l’aide des quantiles extrêmes. Les valeurs situées en dehors de l’intervalle [0.1 %, 99 %] ont été supprimées, en retirant les observations inférieures au premier quantile très bas (q01 = 993.10) et supérieures au 99e percentile (q99 = 9 982.79). Cette approche vise à réduire plus fortement l’impact des outliers tout en conservant la majorité de la structure informative des données. Malgré ce filtrage, la persistance de valeurs extrêmes suggère soit la présence d’erreurs de mesure, soit des cas atypiques réels mais très influents, nécessitant éventuellement un traitement complémentaire (winsorisation, transformation logarithmique ou analyse spécifique des segments extrêmes).

In [ ]:
q01 = data_final["prix_m2"].quantile(0.001)
q99 = data_final["prix_m2"].quantile(0.99)
data_final = data_final[data_final["prix_m2"] > q01]
data_final = data_final[data_final["prix_m2"] < q99]

stats1q = univariate_numeric_analysis(data_final,"prix_m2")

### **densité de population**

In [ ]:
stats3 = univariate_numeric_analysis(data_final,"densite" )

### **Superficie de communes**

In [ ]:
stats4 = univariate_numeric_analysis(data_final,"superficie" )

## **3-2 - Bivariées**

### **3-2-1 -  Variables Quanti/Quanti**

In [ ]:
analyse_bivariée_quant_quant(data_final,["prix_m2", "densite", "taux_chomage", "mediane_niveau_vie"])

L’analyse des corrélations met en évidence plusieurs relations significatives entre le prix au m² et les variables socio-économiques étudiées. Le *prix_m2* est positivement corrélé à la *densité de population* $(r = 0.44)$, ce qui suggère que les zones plus urbanisées tendent à afficher des prix immobiliers plus élevés. Une corrélation positive plus modérée est également observée avec la *médiane du niveau de vie* $(r = 0.34)$, indiquant que les territoires plus aisés sont globalement associés à des prix immobiliers plus élevés. En revanche, la relation entre le *prix_m2* et le *taux de chômage* est quasi nulle $(r ≈ -0.01)$, suggérant une absence de lien linéaire direct dans les données étudiées.

Par ailleurs, les tests de significativité ($p-values$ proches de 0 pour l’ensemble des relations) indiquent que ces corrélations sont statistiquement significatives. On observe également une relation marquée entre les variables explicatives elles-mêmes, notamment une corrélation négative forte entre le *taux de chômage* et la *médiane du niveau de vie* $(r = -0.57)$, traduisant une cohérence socio-économique attendue entre pauvreté relative et revenus médians.

### **3-2-1 -  Variables Quali/Quanti: Cartographie**

**Prix du $m^2$ par departement:**

In [ ]:

carte_france_departements(df=data_final, col="prix_m2")

**Prix du $m^2$  et indicateurs socio-économiques par commune (ou arrondissement) selon departement :**

**Paris 75**

In [ ]:
carte_dep_communes_grid(df=data_final,cols=["prix_m2", "densite", "taux_chomage", "mediane_niveau_vie"],code_dep=75)

**Rhône 69**

In [ ]:
carte_dep_communes_grid(df=data_final,cols=["prix_m2", "densite", "taux_chomage", "mediane_niveau_vie"],code_dep=69)

# **4 - MODÉLISATION**

L'objectif est de prédire le **prix au m²** des biens immobiliers à partir des caractéristiques du bien (DVF) et du contexte socio-économique de la commune (INSEE). Nous comparons trois modèles : une régression linéaire (baseline), un Random Forest et un Gradient Boosting.

### 4.1 - Préparation des données

Nous retenons 7 variables explicatives issues de trois sources. La surface bâtie et le nombre de pièces caractérisent le bien, le type de local distingue maisons et appartements. Côté INSEE, la médiane du niveau de vie et le taux de chômage captent le contexte économique, la densité capte l'effet urbain/rural. Le taux de logements sociaux (data.gouv.fr) caractérise la structure du parc immobilier. Les données sont séparées en 80% entraînement et 20% test (random_state=42 pour la reproductibilité).

In [ ]:
X, y = prepare_features(data_final)
X_train, X_test, y_train, y_test = split_data(X, y)

print(f"Train : {X_train.shape[0]} lignes")
print(f"Test  : {X_test.shape[0]} lignes")
X.dropna()

### 4.2 - Régression linéaire (baseline)

La régression linéaire suppose une relation proportionnelle entre chaque variable et le prix au m². C'est un modèle simple et interprétable grâce à ses coefficients. Il sert de référence .

In [ ]:
reg = train_linear_regression(X_train, y_train)
y_pred_reg = reg.predict(X_test)

res_reg = evaluate_model(y_test, y_pred_reg, "Régression Linéaire")
 

# Coefficients
coefs = pd.DataFrame({
    "variable": X.columns,
    "coefficient": reg.coef_
}).sort_values("coefficient", ascending=False)
coefs

**Interprétation des coefficients :** un coefficient positif pour la médiane du niveau de vie signifie qu'une commune plus riche est associée à un prix au m² plus élevé. À l'inverse, le coefficient négatif de la surface confirme l'effet de dégressivité classique en immobilier : plus un bien est grand, moins le prix au m² est élevé. Le R² faible (≈0.28) montre que la relation entre les prix et les variables n'est pas linéaire.

### 4.3 - Random Forest

Le Random Forest est un ensemble de 100 arbres de décision qui votent ensemble pour la prédiction finale. Contrairement à la régression linéaire, il capture les **relations non-linéaires** et les **interactions entre variables** (par exemple : grande surface dans une commune riche ≠ grande surface dans une commune pauvre).

In [ ]:
rf = train_random_forest(X_train, y_train)
y_pred_rf = rf.predict(X_test)
 
res_rf = evaluate_model(y_test, y_pred_rf, "Random Forest")

### 4.4 - Gradient Boosting

Le Gradient Boosting construit les arbres **séquentiellement** : chaque nouvel arbre corrige les erreurs du précédent. C'est souvent le modèle le plus performant sur des données tabulaires.

In [ ]:
gb = train_gradient_boosting(X_train, y_train)
y_pred_gb = gb.predict(X_test)
 
res_gb = evaluate_model(y_test, y_pred_gb, "Gradient Boosting")
 

### 4.5 - Comparaison des modèles

Trois métriques sont utilisées :
- **MAE** : erreur moyenne absolue en €/m², la plus intuitive
- **RMSE** : pénalise davantage les grosses erreurs, pertinent en immobilier
- **R²** : part de variance expliquée (1 = parfait, 0 = inutile) ( un R² de 0.53 signifie que 53% des différences de prix entre les biens sont captées par nos variables.)

In [ ]:
resultats = pd.DataFrame([res_reg, res_rf, res_gb])
resultats

### 4.6 - Importance des variables

Le Random Forest permet d'identifier les variables qui contribuent le plus à la prédiction du prix au m².

In [ ]:
plot_feature_importance(rf, X.columns)

### 4.7 - Prédictions vs réalité
Ce graphique compare les prix prédits aux prix réels. Plus les points sont proches de la diagonale rouge (prédiction = réalité), meilleur est le modèle. On observe que le Random Forest concentre davantage ses prédictions autour de la diagonale, confirmant sa supériorité.

In [ ]:
plot_predictions(
    y_test,
    [y_pred_reg, y_pred_rf, y_pred_gb],
    ["Régression Linéaire", "Random Forest", "Gradient Boosting"]
)

### 4.9 - Analyse des résidus

La distribution des résidus (erreur = prix réel - prix prédit) permet de vérifier si le modèle a un biais systématique. Si la distribution est centrée autour de 0, le modèle ne sur-estime ni ne sous-estime les prix de manière systématique.
 

In [ ]:

plot_residuals(y_test, y_pred_rf, "Random Forest")

La distribution est centrée autour de 0 (ligne rouge), ce qui confirme l'absence de biais systématique : le modèle ne sur-estime ni ne sous-estime les prix de manière globale. On observe cependant une **asymétrie vers la droite** (queue longue entre 5 000 et 15 000 €/m²) : le modèle sous-estime certains biens chers.


### 4.8 - Tentative d'amélioration : log-transformation

Le prix au m² ayant une distribution asymétrique, nous avons testé une log-transformation : entraîner le modèle sur log(prix) puis retransformer les prédictions. Cette technique permet souvent de mieux capter les variations relatives des prix. Résultat : R² ≈ 0.53, aucun gain par rapport au Random Forest standard. Cela suggère que le plafond de performance est lié aux variables disponibles et non à la méthode de modélisation.

In [ ]:

# Entraîner sur le log du prix
y_train_log = np.log1p(y_train)

rf_log = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_log.fit(X_train, y_train_log)

# Prédire et retransformer
y_pred_log = rf_log.predict(X_test)
y_pred_rf_log = np.expm1(y_pred_log)

# Évaluer sur les vrais prix
mae = mean_absolute_error(y_test, y_pred_rf_log)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf_log))
r2 = r2_score(y_test, y_pred_rf_log)

print(f"=== Random Forest (log) ===")
print(f"MAE  : {mae:.2f} €/m²")
print(f"RMSE : {rmse:.2f} €/m²")
print(f"R²   : {r2:.4f}")

### 4.9 - Analyse par département

Pour comprendre les limites du modèle national, nous testons le Random Forest sur plusieurs départements individuellement.

In [ ]:

for dep in ["35", "69", "33", "59", "13"]:
    df_dep = data_final[data_final["code_departement"] == dep]
    if len(df_dep) > 5000:
        X_dep, y_dep = prepare_features(df_dep)
        X_train_d, X_test_d, y_train_d, y_test_d = split_data(X_dep, y_dep)
        
        rf_dep = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        rf_dep.fit(X_train_d, y_train_d)
        y_pred_d = rf_dep.predict(X_test_d)
        
        print(f"Département {dep} : {len(df_dep)} lignes | R² = {r2_score(y_test_d, y_pred_d):.4f}")

Les performances varient fortement selon le département. Le modèle fonctionne mieux dans des départements au marché diversifié comme l'Ille-et-Vilaine (R² ≈ 0.58) ou la Gironde (R² ≈ 0.56) que dans les départements dominés par une grande métropole comme les Bouches-du-Rhône (R² ≈ 0.25), où les disparités entre quartiers au sein d'une même commune ne sont pas captées par nos variables communales.

### 4.10 - Limites et perspectives

Notre meilleur modèle (Random Forest) explique environ 53% de la variance des prix au m², avec une erreur moyenne de 788 €/m². Nous avons également testé un XGBoost qui n'a pas surpassé le Random Forest sur nos données (R² ≈ 0.47).

Les 47% restants s'expliquent par des caractéristiques absentes de nos données :

- **L'étage** : un appartement au dernier étage avec vue ne vaut pas le même prix qu'un rez-de-chaussée sur cour
- **L'état du bien** : un logement neuf et un logement à rénover de même surface dans la même commune auront des prix très différents
- **La micro-localisation** : nos variables INSEE sont agrégées au niveau communal, or au sein d'une même ville les prix varient fortement d'un quartier à l'autre, comme le montre l'analyse par département (section 4.8)
- **La proximité des transports et commerces** : facteur majeur du prix en zone urbaine, non disponible dans nos sources

Le taux de pauvreté a dû être exclu de la modélisation en raison de 87% de valeurs manquantes, liées au secret statistique appliqué par l'INSEE aux communes de moins de 1 000 ménages.

**Pistes d'amélioration :**
- Utiliser les données INSEE au niveau **IRIS** (infra-communal) pour capter les disparités au sein des communes
- Intégrer les **coordonnées géographiques** (latitude/longitude) des biens pour capter la micro-localisation
- Enrichir avec des données complémentaires : proximité gares, scores d'accessibilité, données cadastrales sur l'ancienneté du bâti